# Azure OpenAI Responses API + Vector Store File Search Demo

This notebook uses **GPT-5.2** on Azure to answer questions about documents stored in a **Vector Store**.

Instead of passing file contents directly in each request, we upload documents into a
Vector Store and use the **`file_search`** tool so the model can retrieve relevant
passages on the fly.

**What you'll see:**
1. **Idempotent vector store creation** — the store is created once; re-running reuses it.
2. **Idempotent batch file upload** — files already in the store are skipped automatically.
3. **Querying with `file_search`** — the model searches the store and cites its sources.

> **Note:** GPT-5.2 requires access approval. Request it at [aka.ms/oai/gpt5access](https://aka.ms/oai/gpt5access).

## Step 0 — Setup

Do these steps **once** in your terminal before opening this notebook.

---

**1. Check that Python is installed**

```bash
python --version
```

You should see `Python 3.11.x` or higher. If not, download it from [python.org](https://www.python.org/downloads/).
On Windows, check **"Add Python to PATH"** during installation.

---

**2. Create a virtual environment**

A virtual environment keeps this project's packages separate from everything else on your machine.

```bash
python -m venv .venv
```

---

**3. Activate the environment**

You'll need to do this every time you open a new terminal for this project.

macOS / Linux:
```bash
source .venv/bin/activate
```

Windows (Command Prompt):
```
.venv\Scripts\activate
```

Your terminal prompt will show `(.venv)` once it's active.

---

**4. Install dependencies**

```bash
pip install -r requirements.txt
```

---

**5. Launch Jupyter**

```bash
jupyter notebook
```

A browser window will open. Click `azure_oai_responses_demo.ipynb` to open this notebook, then run all cells top to bottom.

## Step 1 — Your Azure Credentials & Configuration

Your credentials and settings are loaded from a **`.env`** file so they stay out of version control.

1. Copy the template:  `cp .env.example .env`
2. Open `.env` and fill in your **Endpoint** and **API Key** (found in the Azure Portal under your Azure OpenAI resource -> **Keys and Endpoint**).
3. Set your **deployment name** (found in [Azure AI Foundry](https://oai.azure.com) under **Deployments**).

| Variable | Purpose | Default |
|---|---|---|
| `AZURE_OPENAI_ENDPOINT` | Your Azure OpenAI resource URL | *(required)* |
| `AZURE_OPENAI_API_KEY` | API key for authentication | *(required)* |
| `MODEL_DEPLOYMENT_NAME` | Model deployment name | `gpt-5.2` |
| `REASONING_EFFORT` | How deeply the model reasons (`low`, `medium`, `high`) | `medium` |
| `VECTOR_STORE_NAME` | Name for the vector store (used for idempotent creation) | `demo-vector-store` |
| `DOCUMENTS_DIR` | Directory containing `.txt` files to upload | `documents` |

In [ ]:
from dotenv import dotenv_values
from openai.types.shared import ReasoningEffort

config = dotenv_values(".env")

AZURE_OPENAI_ENDPOINT = config["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY  = config["AZURE_OPENAI_API_KEY"]
MODEL_DEPLOYMENT_NAME = config.get("MODEL_DEPLOYMENT_NAME", "gpt-5.2")
REASONING_EFFORT: ReasoningEffort = config.get("REASONING_EFFORT", "medium")  # type: ignore[assignment]

# Vector store name — used to find or create the store idempotently.
VECTOR_STORE_NAME = config.get("VECTOR_STORE_NAME", "demo-vector-store")

# Directory containing .txt files to upload into the vector store.
DOCUMENTS_DIR = config.get("DOCUMENTS_DIR", "documents")

print(f"Endpoint:     {AZURE_OPENAI_ENDPOINT}")
print(f"Model:        {MODEL_DEPLOYMENT_NAME}")
print(f"Vector Store: {VECTOR_STORE_NAME}")
print(f"Documents:    {DOCUMENTS_DIR}/")

## Step 2 — Connect to Azure OpenAI

We use the standard `openai` Python client pointed at the Azure OpenAI `/openai/v1/` base URL.
This gives us access to the Responses API **and** the Vector Stores API through a single client.

In [ ]:
from openai import OpenAI

# Connect to Azure OpenAI using the standard OpenAI client.
# The base_url points to the Azure-specific /openai/v1/ endpoint which
# provides access to both the Responses API and Vector Stores API.
client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

print("Client connected.")

## Step 3 — Create or Reuse a Vector Store (Idempotent)

A **Vector Store** is a server-side container that holds your documents in an indexed,
searchable form. When you attach it to a query via the `file_search` tool, the model
can retrieve relevant passages automatically.

**Idempotency:** We list all existing vector stores and look for one whose name matches
`VECTOR_STORE_NAME`. If found, we reuse it. If not, we create a new one. This means
you can re-run this cell as many times as you like without creating duplicates.

In [ ]:
def get_or_create_vector_store(name: str) -> str:
    """Find an existing vector store by name, or create a new one.

    This makes the operation idempotent — repeated calls with the same name
    always return the same vector store ID without creating duplicates.

    Args:
        name: The human-readable name to search for / assign to the store.

    Returns:
        The vector store ID (e.g. "vs_abc123...").
    """
    # Search through existing vector stores for a name match
    for vs in client.vector_stores.list():
        if vs.name == name:
            print(f"Found existing vector store '{name}' (ID: {vs.id})")
            return vs.id

    # No match — create a new store
    vs = client.vector_stores.create(name=name)
    print(f"Created new vector store '{name}' (ID: {vs.id})")
    return vs.id

vector_store_id = get_or_create_vector_store(VECTOR_STORE_NAME)

## Step 4 — Upload Files to the Vector Store (Idempotent)

We scan the `DOCUMENTS_DIR` directory for `.txt` files and upload each one to the
vector store. Before uploading, we check which files are **already in the store** by
listing existing vector store files and looking up their filenames. Any file whose
name already exists is skipped.

**How batching works:**
- Each file is first uploaded to Azure's file storage (`client.files.create`).
- The returned file ID is then added to the vector store (`client.vector_stores.files.create_and_poll`).
- `create_and_poll` waits until the file is fully indexed before returning.

This two-step process (upload, then index) is necessary because vector store files
reference file IDs from the general file storage.

In [ ]:
import os


def get_existing_filenames(vs_id: str) -> dict[str, str]:
    """Build a mapping of filename -> file_id for files already in the vector store.

    The vector store file objects only carry a file ID, so we retrieve the
    underlying file object to read its filename.
    """
    existing: dict[str, str] = {}
    for vs_file in client.vector_stores.files.list(vs_id):
        file_obj = client.files.retrieve(vs_file.id)
        existing[file_obj.filename] = vs_file.id
    return existing


def upload_directory(vs_id: str, directory: str) -> list[str]:
    """Upload all .txt files from a directory into the vector store.

    Files whose names already exist in the store are skipped (idempotent).
    Each new file is uploaded to Azure file storage first, then added to the
    vector store and polled until indexing completes.

    Args:
        vs_id: The vector store ID to upload into.
        directory: Local directory path containing .txt files.

    Returns:
        List of newly uploaded file IDs.
    """
    existing = get_existing_filenames(vs_id)
    if existing:
        print(f"Files already in store: {list(existing.keys())}")
    else:
        print("No files in store yet.")

    new_file_ids: list[str] = []
    for filename in sorted(os.listdir(directory)):
        filepath = os.path.join(directory, filename)

        # Only process regular .txt files
        if not os.path.isfile(filepath) or not filename.endswith(".txt"):
            continue

        # Skip files that are already indexed in the vector store
        if filename in existing:
            print(f"  Skipping '{filename}' — already in vector store")
            continue

        # Step 1: Upload the file to Azure file storage
        print(f"  Uploading '{filename}' ...", end=" ", flush=True)
        with open(filepath, "rb") as f:
            uploaded = client.files.create(file=f, purpose="assistants")

        # Step 2: Add the file to the vector store and wait for indexing
        vs_file = client.vector_stores.files.create_and_poll(
            file_id=uploaded.id, vector_store_id=vs_id
        )
        print(f"done (status: {vs_file.status})")
        new_file_ids.append(uploaded.id)

    if not new_file_ids:
        print("Nothing new to upload — all files already indexed.")
    else:
        print(f"\nUploaded {len(new_file_ids)} new file(s).")

    return new_file_ids

uploaded_ids = upload_directory(vector_store_id, DOCUMENTS_DIR)

## Step 5 — Query the Vector Store

Now we ask GPT-5.2 a question. The `file_search` tool tells the model which vector
store to search. The model automatically generates search queries, retrieves relevant
chunks from the indexed documents, and incorporates them into its answer.

In [ ]:
from openai.types.shared_params import Reasoning

response = client.responses.create(
    model=MODEL_DEPLOYMENT_NAME,
    instructions="Answer questions based only on the documents in the vector store. Be concise.",
    reasoning=Reasoning(effort=REASONING_EFFORT),
    max_output_tokens=4000,
    input="Summarize the document, please.",
    tools=[{"type": "file_search", "vector_store_ids": [vector_store_id]}],
)

print(response.output_text)

## Step 6 — Follow-up Question

Let's ask a more specific factual question to confirm the model is actually retrieving
content from our indexed documents.

In [ ]:
response2 = client.responses.create(
    model=MODEL_DEPLOYMENT_NAME,
    instructions="Answer questions based only on the documents in the vector store. Be concise.",
    reasoning=Reasoning(effort=REASONING_EFFORT),
    max_output_tokens=4000,
    input="What is the price of the Pro Plan, and what does it include?",
    tools=[{"type": "file_search", "vector_store_ids": [vector_store_id]}],
)

print(response2.output_text)